In [0]:
%sql
SELECT CASE
  WHEN (SELECT COUNT(*) FROM nyc_mobility.raw.taxi_zones) = 0
    THEN raise_error('raw.taxi_zones is empty, aborting clean run')
  ELSE 'PASS: raw.taxi_zones has ' || (SELECT COUNT(*) FROM nyc_mobility.raw.taxi_zones) || ' rows'
END AS source_check;

In [0]:
%sql
-- Drop rows a failed CAST or blank field would let through silently
CREATE OR REPLACE TEMP VIEW taxi_zones_filtered AS
SELECT * FROM nyc_mobility.raw.taxi_zones
WHERE TRY_CAST(LocationID AS INT) IS NOT NULL
  AND TRIM(COALESCE(CAST(Borough AS STRING), '')) != ''
  AND TRIM(COALESCE(CAST(Zone AS STRING), '')) != '';

SELECT
  (SELECT COUNT(*) FROM nyc_mobility.raw.taxi_zones) AS rows_before,
  (SELECT COUNT(*) FROM taxi_zones_filtered) AS rows_after,
  (SELECT COUNT(*) FROM nyc_mobility.raw.taxi_zones) -
  (SELECT COUNT(*) FROM taxi_zones_filtered) AS rows_excluded;

In [0]:
%sql
CREATE OR REPLACE TABLE nyc_mobility.clean.taxi_zones AS
SELECT
    CAST(LocationID AS INT) AS location_id,
    CASE WHEN UPPER(TRIM(CAST(Borough AS STRING))) = 'EWR'
         THEN UPPER(TRIM(CAST(Borough AS STRING)))
         ELSE INITCAP(TRIM(CAST(Borough AS STRING))) END AS borough,
    CASE WHEN UPPER(TRIM(CAST(Zone AS STRING))) = 'SOHO' THEN 'SoHo'
         WHEN UPPER(TRIM(CAST(Zone AS STRING))) = 'LAGUARDIA AIRPORT' THEN 'LaGuardia Airport'
         WHEN UPPER(TRIM(CAST(Zone AS STRING))) = 'JFK AIRPORT' THEN 'JFK Airport'
         WHEN UPPER(TRIM(CAST(Zone AS STRING))) = 'DUMBO/VINEGAR HILL' THEN 'DUMBO/Vinegar Hill'
         ELSE INITCAP(TRIM(CAST(Zone AS STRING))) END AS zone,
    CASE WHEN UPPER(TRIM(CAST(service_zone AS STRING))) = 'EWR'
         THEN UPPER(TRIM(CAST(service_zone AS STRING)))
         ELSE INITCAP(TRIM(CAST(service_zone AS STRING))) END AS service_zone
FROM taxi_zones_filtered;

In [0]:
%sql
-- Write validation, row parity plus primary key integrity
SELECT CASE
  WHEN (SELECT COUNT(*) FROM taxi_zones_filtered) != (SELECT COUNT(*) FROM nyc_mobility.clean.taxi_zones)
    THEN raise_error('Row count mismatch after write')
  WHEN (SELECT COUNT(*) FROM nyc_mobility.clean.taxi_zones) !=
       (SELECT COUNT(DISTINCT location_id) FROM nyc_mobility.clean.taxi_zones)
    THEN raise_error('location_id is not unique')
  WHEN (SELECT COUNT(*) FROM nyc_mobility.clean.taxi_zones WHERE location_id IS NULL) > 0
    THEN raise_error('NULL location_id found')
  ELSE 'PASS: ' || (SELECT COUNT(*) FROM nyc_mobility.clean.taxi_zones) || ' rows, location_id unique and non-null'
END AS write_check;

In [0]:
%sql
-- Only shows a populated result if Cells 1-4 all passed
SELECT * FROM nyc_mobility.clean.taxi_zones
ORDER BY location_id
LIMIT 20;